# Tutorial: Rare Variant Association Testing with `cellink`

This tutorial demonstrates how to perform rare variant association testing (RVAT) in single-cell data using the `cellink` package. We focus on two major types of RVAT: burden tests and the Sequence Kernel Association Test (SKAT). Rare variants can be especially informative for understanding disease biology, but they require careful aggregation and testing approaches due to their low frequency. 

We’ll walk through how to:
- Prepare single-cell and genotype data using `HiData` objects.
- Apply burden tests with multiple variant annotation-based weights.
- Combine p-values from multiple tests using ACAT.
- Perform SKAT for gene-level rare variant analysis.

This tutorial uses a subset of data from the 1k1k project, filtered to the **CD8 Naive** T cell type, and focuses on **chromosome 22** for runtime feasibility. It builds on earlier tutorials that cover pseudobulk expression and variant annotation, which should be reviewed for full context.


> Note: This notebook assumes that you have already run the variant annotation step as described in the [Variant Annotation Tutorial](./explore_annotations.ipynb).

> Prerequisites: To run this notebook, you need to install the required dependencies for RVAT. Use the following command to install the necessary dependencies:
```bash
pip install -e "cellink[rvat, datasets]"
conda install -c conda-forge chiscore

## Environment Setup

We begin by importing necessary libraries and defining key parameters for our analysis. `cellink` provides utilities that extend `AnnData` to handle both donor-level genotype data and single-cell RNA-seq data efficiently.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd

In [3]:
from pathlib import Path
import warnings

import anndata as ad
import scanpy as sc
import dask.array as da
import numpy as np
from tqdm.auto import tqdm

import cellink as cl
from cellink._core import DAnn
from cellink.tl._rvat import run_burden_test, run_skat_test, beta_weighting
from cellink.utils import column_normalize, gaussianize
from cellink.at.acat import acat_test
from cellink.resources import get_dummy_onek1k

site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [4]:
DATA = Path(cl.__file__).parent.parent.parent / "docs/tutorials/data"

In [5]:
n_gpcs = 20
n_epcs = 15
batch_e_pcs_n_top_genes = 2000
chrom = 22
cis_window = 100_000
cell_type = "CD4 Naive"
pb_gex_key = f"PB_{cell_type}"  # pseudobulk expression in hd.G.obsm[key_added]
original_donor_col = "donor_id"
min_percent_donors_expressed = 0.1
celltype_key = "predicted.celltype.l2"
do_debug = False

## Load and Prepare Data

Here, we load a prepared dataset (`onek1k`) that includes genotype and expression information from human donors. (This is a subset of the full OneK1K dataset, which can be downloaded, and prepared using `get_onek1k()`) We also extract gene annotations using Ensembl via `pybiomart`, which are essential for defining cis-windows during eQTL analysis.

In [6]:
hd = get_dummy_onek1k(verify_checksum=False)
hd

[2026-11-24 03:55:31,582] INFO:root: ~/cellink_data/dummy_onek1k/dummy_onek1k_dataset.hd.h5 already exists


[2026-11-24 03:55:31,583] WARNING:root: No checksum provided, skipping verification


[2026-11-24 03:55:45,967] INFO:root: Loaded dummy OneK1K dataset: (300, 146939, 125913, 34830)


╔═ HiData(n_groups=300, n_cells=125,913) ══════════════════════════════════════╗
║ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓ ║
║ ┃ G (groups)                         ┃ C (cells, 333..420 per group)       ┃ ║
║ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩ ║
║ │ AnnData object with n_obs × n_vars │ AnnData object with n_obs × n_vars  │ ║
║ │ = 300 × 146,939                    │ = 125,913 × 34,830                  │ ║
║ │     var: 'AF', 'AN', 'ER2',        │     obs: 'orig.ident',              │ ║
║ │ 'IMPUTED', 'maf', 'NS', 'R2',      │ 'nCount_RNA', 'nFeature_RNA',       │ ║
║ │ 'TYPED', 'TYPED_ONLY', 'id',       │ 'percent.mt', 'donor_id',           │ ║
║ │ 'id_mask', 'length', 'pos',        │ 'pool_number',                      │ ║
║ │ 'quality', 'a0', 'a1', 'chrom',    │ 'predicted.celltype.l2',            │ ║
║ │ 'pos_hg19', 'id_hg19'              │ 'predicted.celltype.l2.score',      │ ║
║ │                                    │ 'age', 'organism_ontology_term_id', │ ║
║ │                                    │ 'tissue_ontology_term_id',          │ ║
║ │                                    │ 'assay_ontology_term_id',           │ ║
║ │                                    │ 'disease_ontology_term_id',         │ ║
║ │                                    │ 'cell_type_ontology_term_id',       │ ║
║ │                                    │ 'self_reported_ethnicity_ontology_… │ ║
║ │                                    │ 'development_stage_ontology_term_i… │ ║
║ │                                    │ 'sex_ontology_term_id',             │ ║
║ │                                    │ 'is_primary_data',                  │ ║
║ │                                    │ 'suspension_type', 'tissue_type',   │ ║
║ │                                    │ 'cell_type', 'assay', 'disease',    │ ║
║ │                                    │ 'organism', 'sex', 'tissue',        │ ║
║ │                                    │ 'self_reported_ethnicity',          │ ║
║ │                                    │ 'development_stage',                │ ║
║ │                                    │ 'observation_joinid'                │ ║
║ │     uns: 'dummy_onek1k', 'kinship' │     var: 'vst.mean',                │ ║
║ │                                    │ 'vst.variance',                     │ ║
║ │                                    │ 'vst.variance.expected',            │ ║
║ │                                    │ 'vst.variance.standardized',        │ ║
║ │                                    │ 'vst.variable',                     │ ║
║ │                                    │ 'feature_is_filtered',              │ ║
║ │                                    │ 'feature_name',                     │ ║
║ │                                    │ 'feature_reference',                │ ║
║ │                                    │ 'feature_biotype',                  │ ║
║ │                                    │ 'feature_length', 'feature_type',   │ ║
║ │                                    │ 'start', 'end', 'chrom', 'strand'   │ ║
║ │     obsm: 'gPCs'                   │     uns:                            │ ║
║ │                                    │ 'cell_type_ontology_term_id_colors… │ ║
║ │                                    │ 'citation', 'default_embedding',    │ ║
║ │                                    │ 'schema_reference',                 │ ║
║ │                                    │ 'schema_version', 'title'           │ ║
║ │                                    │     obsm: 'X_azimuth_spca',         │ ║
║ │                                    │ 'X_azimuth_umap', 'X_harmony',      │ ║
║ │                                    │ 'X_pca', 'X_umap'                   │ ║
║ │                                    │     varm: 'PCs'                     │ ║
║ └────────────────────────────────────┴─────────────────────────────────────┘ ║
╚═ group_id='donor_id' ════════════════════════════════════════════════════════╝

In [7]:
"""
def _get_ensembl_gene_id_start_end_chr():
    from pybiomart import Server

    server = Server(host="http://www.ensembl.org")
    dataset = server.marts["ENSEMBL_MART_ENSEMBL"].datasets["hsapiens_gene_ensembl"]
    ensembl_gene_id_start_end_chr = dataset.query(
        attributes=["ensembl_gene_id", "start_position", "end_position", "chromosome_name"]
    )
    ensembl_gene_id_start_end_chr = ensembl_gene_id_start_end_chr.set_index("Gene stable ID")
    ensembl_gene_id_start_end_chr = ensembl_gene_id_start_end_chr.rename(
        columns={
            "Gene start (bp)": GAnn.start,
            "Gene end (bp)": GAnn.end,
            "Chromosome/scaffold name": GAnn.chrom,
        }
    )
    return ensembl_gene_id_start_end_chr
"""

'\ndef _get_ensembl_gene_id_start_end_chr():\n    from pybiomart import Server\n\n    server = Server(host="http://www.ensembl.org")\n    dataset = server.marts["ENSEMBL_MART_ENSEMBL"].datasets["hsapiens_gene_ensembl"]\n    ensembl_gene_id_start_end_chr = dataset.query(\n        attributes=["ensembl_gene_id", "start_position", "end_position", "chromosome_name"]\n    )\n    ensembl_gene_id_start_end_chr = ensembl_gene_id_start_end_chr.set_index("Gene stable ID")\n    ensembl_gene_id_start_end_chr = ensembl_gene_id_start_end_chr.rename(\n        columns={\n            "Gene start (bp)": GAnn.start,\n            "Gene end (bp)": GAnn.end,\n            "Chromosome/scaffold name": GAnn.chrom,\n        }\n    )\n    return ensembl_gene_id_start_end_chr\n'

In [8]:
"""
ensembl_gene_id_start_end_chr = _get_ensembl_gene_id_start_end_chr()
ensembl_gene_id_start_end_chr
"""

'\nensembl_gene_id_start_end_chr = _get_ensembl_gene_id_start_end_chr()\nensembl_gene_id_start_end_chr\n'

In [9]:
"""
hd.C.var = hd.C.var.join(ensembl_gene_id_start_end_chr)
hd.C.obs[DAnn.donor] = hd.C.obs[original_donor_col]
"""
hd.G.obsm["gPCs"] = hd.G.obsm["gPCs"][hd.G.obsm["gPCs"].columns[:n_gpcs]]

## Normalize and Filter Expression Data

We normalize single-cell expression using total-count normalization and log transformation. We then filter the dataset to only include cells of a specific type (`CD8 Naive`). This step ensures we're analyzing homogeneous populations, which increases power in eQTL detection.

In [10]:
sc.pp.normalize_total(hd.C)
sc.pp.log1p(hd.C)
sc.pp.normalize_total(hd.C)

# are the expression pcs computed by pseudobulking across all cell types?
mdata = sc.get.aggregate(hd.C, by=DAnn.donor, func="mean")
mdata.X = mdata.layers.pop("mean")

sc.pp.highly_variable_genes(mdata, n_top_genes=batch_e_pcs_n_top_genes)
sc.tl.pca(mdata, n_comps=n_epcs)

hd.G.obsm["ePCs"] = mdata[hd.G.obs_names].obsm["X_pca"]

In [11]:
hd = hd[..., hd.C.obs[celltype_key] == cell_type, :].copy()
hd

╔═ HiData(n_groups=300, n_cells=26,230) ═══════════════════════════════════════╗
║ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓ ║
║ ┃ G (groups)                         ┃ C (cells, 2..236 per group)         ┃ ║
║ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩ ║
║ │ AnnData object with n_obs × n_vars │ AnnData object with n_obs × n_vars  │ ║
║ │ = 300 × 146,939                    │ = 26,230 × 34,830                   │ ║
║ │     var: 'AF', 'AN', 'ER2',        │     obs: 'orig.ident',              │ ║
║ │ 'IMPUTED', 'maf', 'NS', 'R2',      │ 'nCount_RNA', 'nFeature_RNA',       │ ║
║ │ 'TYPED', 'TYPED_ONLY', 'id',       │ 'percent.mt', 'donor_id',           │ ║
║ │ 'id_mask', 'length', 'pos',        │ 'pool_number',                      │ ║
║ │ 'quality', 'a0', 'a1', 'chrom',    │ 'predicted.celltype.l2',            │ ║
║ │ 'pos_hg19', 'id_hg19'              │ 'predicted.celltype.l2.score',      │ ║
║ │                                    │ 'age', 'organism_ontology_term_id', │ ║
║ │                                    │ 'tissue_ontology_term_id',          │ ║
║ │                                    │ 'assay_ontology_term_id',           │ ║
║ │                                    │ 'disease_ontology_term_id',         │ ║
║ │                                    │ 'cell_type_ontology_term_id',       │ ║
║ │                                    │ 'self_reported_ethnicity_ontology_… │ ║
║ │                                    │ 'development_stage_ontology_term_i… │ ║
║ │                                    │ 'sex_ontology_term_id',             │ ║
║ │                                    │ 'is_primary_data',                  │ ║
║ │                                    │ 'suspension_type', 'tissue_type',   │ ║
║ │                                    │ 'cell_type', 'assay', 'disease',    │ ║
║ │                                    │ 'organism', 'sex', 'tissue',        │ ║
║ │                                    │ 'self_reported_ethnicity',          │ ║
║ │                                    │ 'development_stage',                │ ║
║ │                                    │ 'observation_joinid'                │ ║
║ │     uns: 'dummy_onek1k', 'kinship' │     var: 'vst.mean',                │ ║
║ │                                    │ 'vst.variance',                     │ ║
║ │                                    │ 'vst.variance.expected',            │ ║
║ │                                    │ 'vst.variance.standardized',        │ ║
║ │                                    │ 'vst.variable',                     │ ║
║ │                                    │ 'feature_is_filtered',              │ ║
║ │                                    │ 'feature_name',                     │ ║
║ │                                    │ 'feature_reference',                │ ║
║ │                                    │ 'feature_biotype',                  │ ║
║ │                                    │ 'feature_length', 'feature_type',   │ ║
║ │                                    │ 'start', 'end', 'chrom', 'strand'   │ ║
║ │     obsm: 'gPCs', 'ePCs'           │     uns:                            │ ║
║ │                                    │ 'cell_type_ontology_term_id_colors… │ ║
║ │                                    │ 'citation', 'default_embedding',    │ ║
║ │                                    │ 'schema_reference',                 │ ║
║ │                                    │ 'schema_version', 'title', 'log1p'  │ ║
║ │                                    │     obsm: 'X_azimuth_spca',         │ ║
║ │                                    │ 'X_azimuth_umap', 'X_harmony',      │ ║
║ │                                    │ 'X_pca', 'X_umap'                   │ ║
║ │                                    │     varm: 'PCs'                     │ ║
║ └────────────────────────────────────┴─────────────────────────────────────┘ ║
╚═ group_id='donor_id' ════════════════════════════════════════════════════════╝

In [12]:
hd.aggregate(key_added=pb_gex_key, sync_var=True, verbose=True)
hd.aggregate(obs=["sex", "age"], func="first", add_to_obs=True)
hd

[2026-11-24 03:56:28,326] INFO:hidata._core.hidata: Aggregated X to PB_CD4 Naive, observed for 300 groups.


╔═ HiData(n_groups=300, n_cells=26,230) ═══════════════════════════════════════╗
║ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓ ║
║ ┃ G (groups)                         ┃ C (cells, 2..236 per group)         ┃ ║
║ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩ ║
║ │ AnnData object with n_obs × n_vars │ AnnData object with n_obs × n_vars  │ ║
║ │ = 300 × 146,939                    │ = 26,230 × 34,830                   │ ║
║ │     obs: 'sex', 'age'              │     obs: 'orig.ident',              │ ║
║ │                                    │ 'nCount_RNA', 'nFeature_RNA',       │ ║
║ │                                    │ 'percent.mt', 'donor_id',           │ ║
║ │                                    │ 'pool_number',                      │ ║
║ │                                    │ 'predicted.celltype.l2',            │ ║
║ │                                    │ 'predicted.celltype.l2.score',      │ ║
║ │                                    │ 'age', 'organism_ontology_term_id', │ ║
║ │                                    │ 'tissue_ontology_term_id',          │ ║
║ │                                    │ 'assay_ontology_term_id',           │ ║
║ │                                    │ 'disease_ontology_term_id',         │ ║
║ │                                    │ 'cell_type_ontology_term_id',       │ ║
║ │                                    │ 'self_reported_ethnicity_ontology_… │ ║
║ │                                    │ 'development_stage_ontology_term_i… │ ║
║ │                                    │ 'sex_ontology_term_id',             │ ║
║ │                                    │ 'is_primary_data',                  │ ║
║ │                                    │ 'suspension_type', 'tissue_type',   │ ║
║ │                                    │ 'cell_type', 'assay', 'disease',    │ ║
║ │                                    │ 'organism', 'sex', 'tissue',        │ ║
║ │                                    │ 'self_reported_ethnicity',          │ ║
║ │                                    │ 'development_stage',                │ ║
║ │                                    │ 'observation_joinid'                │ ║
║ │     var: 'AF', 'AN', 'ER2',        │     var: 'vst.mean',                │ ║
║ │ 'IMPUTED', 'maf', 'NS', 'R2',      │ 'vst.variance',                     │ ║
║ │ 'TYPED', 'TYPED_ONLY', 'id',       │ 'vst.variance.expected',            │ ║
║ │ 'id_mask', 'length', 'pos',        │ 'vst.variance.standardized',        │ ║
║ │ 'quality', 'a0', 'a1', 'chrom',    │ 'vst.variable',                     │ ║
║ │ 'pos_hg19', 'id_hg19'              │ 'feature_is_filtered',              │ ║
║ │                                    │ 'feature_name',                     │ ║
║ │                                    │ 'feature_reference',                │ ║
║ │                                    │ 'feature_biotype',                  │ ║
║ │                                    │ 'feature_length', 'feature_type',   │ ║
║ │                                    │ 'start', 'end', 'chrom', 'strand'   │ ║
║ │     uns: 'dummy_onek1k', 'kinship' │     uns:                            │ ║
║ │                                    │ 'cell_type_ontology_term_id_colors… │ ║
║ │                                    │ 'citation', 'default_embedding',    │ ║
║ │                                    │ 'schema_reference',                 │ ║
║ │                                    │ 'schema_version', 'title', 'log1p'  │ ║
║ │     obsm: 'gPCs', 'ePCs', 'PB_CD4  │     obsm: 'X_azimuth_spca',         │ ║
║ │ Naive'                             │ 'X_azimuth_umap', 'X_harmony',      │ ║
║ │                                    │ 'X_pca', 'X_umap'                   │ ║
║ │                                    │     varm: 'PCs'                     │ ║
║ └────────────────────────────────────┴─────────────────────────────────────┘ ║
╚═ group_id='donor_id' ════════════════════════════════════════════════════════╝

In [13]:
print(f"{pb_gex_key} shape:", hd.G.obsm[pb_gex_key].shape)
print("hd.shape:", hd.shape)

keep_genes = ((hd.G.obsm[pb_gex_key] > 0).mean(axis=0) >= min_percent_donors_expressed).values
hd = hd[..., keep_genes]
print("after filtering")
print(f"{pb_gex_key} shape:", hd.G.obsm[pb_gex_key].shape)
print("hd.shape:", hd.shape)

PB_CD4 Naive shape: (300, 34830)
hd.shape: (300, 146939, 26230, 34830)


after filtering
PB_CD4 Naive shape: (300, 12934)
hd.shape: (300, 146939, 26230, 12934)


In [14]:
# alternative to hd[:, hd.G.var.chrom == str(chrom), :, hd.C.var.chrom == str(chrom)]
hd = hd.sel(G_var=hd.G.var.chrom == str(chrom), C_var=hd.C.var.chrom == str(chrom)).copy()
hd

╔═ HiData(n_groups=300, n_cells=26,230) ═══════════════════════════════════════╗
║ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓ ║
║ ┃ G (groups)                         ┃ C (cells, 2..236 per group)         ┃ ║
║ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩ ║
║ │ AnnData object with n_obs × n_vars │ AnnData object with n_obs × n_vars  │ ║
║ │ = 300 × 136,776                    │ = 26,230 × 351                      │ ║
║ │     obs: 'sex', 'age'              │     obs: 'orig.ident',              │ ║
║ │                                    │ 'nCount_RNA', 'nFeature_RNA',       │ ║
║ │                                    │ 'percent.mt', 'donor_id',           │ ║
║ │                                    │ 'pool_number',                      │ ║
║ │                                    │ 'predicted.celltype.l2',            │ ║
║ │                                    │ 'predicted.celltype.l2.score',      │ ║
║ │                                    │ 'age', 'organism_ontology_term_id', │ ║
║ │                                    │ 'tissue_ontology_term_id',          │ ║
║ │                                    │ 'assay_ontology_term_id',           │ ║
║ │                                    │ 'disease_ontology_term_id',         │ ║
║ │                                    │ 'cell_type_ontology_term_id',       │ ║
║ │                                    │ 'self_reported_ethnicity_ontology_… │ ║
║ │                                    │ 'development_stage_ontology_term_i… │ ║
║ │                                    │ 'sex_ontology_term_id',             │ ║
║ │                                    │ 'is_primary_data',                  │ ║
║ │                                    │ 'suspension_type', 'tissue_type',   │ ║
║ │                                    │ 'cell_type', 'assay', 'disease',    │ ║
║ │                                    │ 'organism', 'sex', 'tissue',        │ ║
║ │                                    │ 'self_reported_ethnicity',          │ ║
║ │                                    │ 'development_stage',                │ ║
║ │                                    │ 'observation_joinid'                │ ║
║ │     var: 'AF', 'AN', 'ER2',        │     var: 'vst.mean',                │ ║
║ │ 'IMPUTED', 'maf', 'NS', 'R2',      │ 'vst.variance',                     │ ║
║ │ 'TYPED', 'TYPED_ONLY', 'id',       │ 'vst.variance.expected',            │ ║
║ │ 'id_mask', 'length', 'pos',        │ 'vst.variance.standardized',        │ ║
║ │ 'quality', 'a0', 'a1', 'chrom',    │ 'vst.variable',                     │ ║
║ │ 'pos_hg19', 'id_hg19'              │ 'feature_is_filtered',              │ ║
║ │                                    │ 'feature_name',                     │ ║
║ │                                    │ 'feature_reference',                │ ║
║ │                                    │ 'feature_biotype',                  │ ║
║ │                                    │ 'feature_length', 'feature_type',   │ ║
║ │                                    │ 'start', 'end', 'chrom', 'strand'   │ ║
║ │     uns: 'dummy_onek1k', 'kinship' │     uns:                            │ ║
║ │                                    │ 'cell_type_ontology_term_id_colors… │ ║
║ │                                    │ 'citation', 'default_embedding',    │ ║
║ │                                    │ 'schema_reference',                 │ ║
║ │                                    │ 'schema_version', 'title', 'log1p'  │ ║
║ │     obsm: 'gPCs', 'ePCs', 'PB_CD4  │     obsm: 'X_azimuth_spca',         │ ║
║ │ Naive'                             │ 'X_azimuth_umap', 'X_harmony',      │ ║
║ │                                    │ 'X_pca', 'X_umap'                   │ ║
║ │                                    │     varm: 'PCs'                     │ ║
║ └────────────────────────────────────┴─────────────────────────────────────┘ ║
╚═ group_id='donor_id' ════════════════════════════════════════════════════════╝

## Step 1: Annotate Variants with Functional Information

To inform the weighting in burden tests, we annotate variants using the VEP tool. These annotations include predicted functional consequences (e.g., missense, stop gained), CADD scores, and distance to gene transcription start sites (TSS).

This information allows biologically meaningful prioritization of variants when aggregating their effects.

In [15]:
vep_annotation_file = DATA / "variant_annotation/variants_vep_annotated.txt"

In [16]:
cl.tl.add_vep_annos_to_gdata(vep_anno_file=vep_annotation_file, gdata=hd.G, dummy_consequence=True)
hd.G.uns["variant_annotation_vep"]

[2026-11-24 03:56:32,855] INFO:cellink.tl._annotate_snps_genotype_data: Preparing VEP annotations for addition to gdata


[2026-11-24 03:56:32,856] INFO:cellink.tl._annotate_snps_genotype_data: Reading annotation file docs/tutorials/data/variant_annotation/variants_vep_annotated.txt


[2026-11-24 03:56:34,044] INFO:cellink.tl._annotate_snps_genotype_data: Annotation file loaded


[2026-11-24 03:56:34,133] INFO:cellink.tl._annotate_snps_genotype_data: Annotation columns: ['snp_id', 'Location', 'Allele', 'gene_id', 'transcript_id', 'Feature_type', 'Consequence', 'cDNA_position', 'CDS_position', 'Protein_position', 'Amino_acids', 'Codons', 'Existing_variation', 'IMPACT', 'DISTANCE', 'STRAND', 'FLAGS', 'BIOTYPE', 'CANONICAL', 'ENSP', 'SIFT', 'PolyPhen', 'gnomADe_AF', 'gnomADe_AFR_AF', 'gnomADe_AMR_AF', 'gnomADe_ASJ_AF', 'gnomADe_EAS_AF', 'gnomADe_FIN_AF', 'gnomADe_NFE_AF', 'gnomADe_OTH_AF', 'gnomADe_SAS_AF', 'CLIN_SIG', 'SOMATIC', 'PHENO', 'CADD_PHRED', 'CADD_RAW', 'TSSDistance']


[2026-11-24 03:56:39,555] INFO:cellink.tl._annotate_snps_genotype_data: Changing dtype of categorical columns ['snp_id', 'gene_id', 'transcript_id', 'cDNA_position', 'BIOTYPE', 'gnomADe_AFR_AF', 'CADD_PHRED', 'STRAND', 'PHENO', 'Amino_acids', 'gnomADe_EAS_AF', 'IMPACT', 'SOMATIC', 'Codons', 'CLIN_SIG', 'PolyPhen', 'CDS_position', 'Protein_position', 'Existing_variation', 'ENSP', 'gnomADe_FIN_AF', 'gnomADe_AF', 'SIFT', 'gnomADe_OTH_AF', 'DISTANCE', 'gnomADe_SAS_AF', 'gnomADe_AMR_AF', 'gnomADe_ASJ_AF', 'FLAGS', 'gnomADe_NFE_AF', 'CANONICAL', 'CADD_RAW', 'TSSDistance']


[2026-11-24 03:56:41,370] INFO:cellink.tl._annotate_snps_genotype_data: VEP annotations added to gdata


Consequence_3_prime_UTR_variant  \
snp_id          gene_id         transcript_id                                      
22_16388891_G_A ENSG00000225293 ENST00000399687                                0   
22_16388968_C_T ENSG00000225293 ENST00000399687                                0   
22_16389525_A_G ENSG00000225293 ENST00000399687                                0   
22_16390411_G_A ENSG00000225293 ENST00000399687                                0   
22_16391555_G_C ENSG00000225293 ENST00000399687                                0   
...                                                                          ...   
22_50798635_T_C ENSG00000291064 ENST00000480246                                0   
22_50799821_A_C ENSG00000291064 ENST00000480246                                0   
                ENSG00000184319 ENST00000427528                                0   
22_50802428_C_G ENSG00000184319 ENST00000427528                                0   
                ENSG00000291064 ENST00000480246                                0   

                                                 Consequence_5_prime_UTR_variant  \
snp_id          gene_id         transcript_id                                      
22_16388891_G_A ENSG00000225293 ENST00000399687                                0   
22_16388968_C_T ENSG00000225293 ENST00000399687                                0   
22_16389525_A_G ENSG00000225293 ENST00000399687                                0   
22_16390411_G_A ENSG00000225293 ENST00000399687                                0   
22_16391555_G_C ENSG00000225293 ENST00000399687                                0   
...                                                                          ...   
22_50798635_T_C ENSG00000291064 ENST00000480246                                0   
22_50799821_A_C ENSG00000291064 ENST00000480246                                0   
                ENSG00000184319 ENST00000427528                                0   
22_50802428_C_G ENSG00000184319 ENST00000427528                                0   
                ENSG00000291064 ENST00000480246                                0   

                                                 Consequence_NMD_transcript_variant  \
snp_id          gene_id         transcript_id                                         
22_16388891_G_A ENSG00000225293 ENST00000399687                                   0   
22_16388968_C_T ENSG00000225293 ENST00000399687                                   0   
22_16389525_A_G ENSG00000225293 ENST00000399687                                   0   
22_16390411_G_A ENSG00000225293 ENST00000399687                                   0   
22_16391555_G_C ENSG00000225293 ENST00000399687                                   0   
...                                                                             ...   
22_50798635_T_C ENSG00000291064 ENST00000480246                                   0   
22_50799821_A_C ENSG00000291064 ENST00000480246                                   0   
                ENSG00000184319 ENST00000427528                                   0   
22_50802428_C_G ENSG00000184319 ENST00000427528                                   0   
                ENSG00000291064 ENST00000480246                                   0   

                                                 Consequence_coding_sequence_variant  \
snp_id          gene_id         transcript_id                                          
22_16388891_G_A ENSG00000225293 ENST00000399687                                    0   
22_16388968_C_T ENSG00000225293 ENST00000399687                                    0   
22_16389525_A_G ENSG00000225293 ENST00000399687                                    0   
22_16390411_G_A ENSG00000225293 ENST00000399687                                    0   
22_16391555_G_C ENSG00000225293 ENST00000399687                                    0   
...                                                                              ...   
22_50798635_T_C ENSG0000029

In [17]:
cl.tl.aggregate_annotations_for_varm(
    hd.G, "variant_annotation_vep", agg_type="first", return_data=True
)  # TODO change agg type

[2026-11-24 03:56:41,696] INFO:cellink.tl._annotate_snps_genotype_data: Aggregating using method: first


,gene_id,transcript_id,Consequence_3_prime_UTR_variant,Consequence_5_prime_UTR_variant,Consequence_NMD_transcript_variant,Consequence_coding_sequence_variant,Consequence_downstream_gene_variant,Consequence_frameshift_variant,Consequence_incomplete_terminal_codon_variant,Consequence_inframe_deletion,...,gnomADe_OTH_AF,DISTANCE,gnomADe_SAS_AF,gnomADe_AMR_AF,gnomADe_ASJ_AF,FLAGS,gnomADe_NFE_AF,CANONICAL,CADD_RAW,TSSDistance
snp_id,,,,,,,,,,,,,,,,,,,,,
22_16388891_G_A,ENSG00000225293,ENST00000399687,0,0,0,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,-,NaN,YES,0.712538,NaN
22_16388968_C_T,ENSG00000225293,ENST00000399687,0,0,0,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,-,NaN,YES,0.670150,NaN
22_16389525_A_G,ENSG00000225293,ENST00000399687,0,0,0,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,-,NaN,YES,0.559458,NaN
22_16390411_G_A,ENSG00000225293,ENST00000399687,0,0,0,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,-,NaN,YES,1.787856,NaN
22_16391555_G_C,ENSG00000225293,ENST00000399687,0,0,0,0,1,0,0,0,...,NaN,668.0,NaN,NaN,NaN,-,NaN,YES,0.137342,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22_50796327_CCA_C,ENSG00000291064,ENST00000480246,0,0,0,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,-,NaN,-,-0.211274,NaN
22_50798021_A_G,ENSG00000184319,ENST00000427528,0,0,0,0,0,0,0,0,...,NaN,634.0,NaN,NaN,NaN,-,NaN,YES,0.478246,634.0
22_50798635_T_C,ENSG00000184319,ENST00000427528,0,0,0,0,0,0,0,0,...,NaN,20.0,NaN,NaN,NaN,-,NaN,YES,0.363428,20.0


In [18]:
hd.G.varm["variant_annotation"].columns

Index(['gene_id', 'transcript_id', 'Consequence_3_prime_UTR_variant',
       'Consequence_5_prime_UTR_variant', 'Consequence_NMD_transcript_variant',
       'Consequence_coding_sequence_variant',
       'Consequence_downstream_gene_variant', 'Consequence_frameshift_variant',
       'Consequence_incomplete_terminal_codon_variant',
       'Consequence_inframe_deletion', 'Consequence_inframe_insertion',
       'Consequence_intergenic_variant', 'Consequence_intron_variant',
       'Consequence_mature_miRNA_variant', 'Consequence_missense_variant',
       'Consequence_non_coding_transcript_exon_variant',
       'Consequence_non_coding_transcript_variant',
       'Consequence_splice_acceptor_variant',
       'Consequence_splice_donor_5th_base_variant',
       'Consequence_splice_donor_region_variant',
       'Consequence_splice_donor_variant',
       'Consequence_splice_polypyrimidine_tract_variant',
       'Consequence_splice_region_variant', 'Consequence_stop_gained',
       'Consequence_s

In [19]:
hd.G.uns["variant_annotation_vep"]["CADD_RAW"].describe()

count    178574.000000
mean          0.080857
std           0.497543
min          -3.708266
25%          -0.169380
50%           0.053264
75%           0.299364
max           7.465009
Name: CADD_RAW, dtype: float64

## Step 2: Burden Testing

We now perform burden tests, which evaluate whether the **aggregate effect** of rare variants within a gene is associated with a phenotype, in this case gene expression in CD8 Naive cells. We use different weighting schemes allow us to prioritize variants differently based on functional scores or allele frequency, including:
- **CADD_RAW**: Raw CADD scores.
- **maf_beta**: Beta weighting based on minor allele frequency (MAF).
- **tss_distance**: Distance to the transcription start site (TSS).
- **tss_distance_exp**: Exponential weighting based on TSS distance.

In [20]:
burden_agg_fct = "sum"
run_lrt = True
annotation_cols = ["CADD_RAW", "maf_beta", "tss_distance", "tss_distance_exp"]

rare_maf_threshold = 0.05

### Filtering for Rare Variants
Burden tests focus on rare variants, typically those with minor allele frequency (MAF) below 0.01 or 0.05. We filter our genotype data accordingly to isolate variants in this frequency range.

In [21]:
hd = hd.sel(G_var=hd.G.var.maf < rare_maf_threshold).copy()
hd

╔═ HiData(n_groups=300, n_cells=26,230) ═══════════════════════════════════════╗
║ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓ ║
║ ┃ G (groups)                         ┃ C (cells, 2..236 per group)         ┃ ║
║ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩ ║
║ │ AnnData object with n_obs × n_vars │ AnnData object with n_obs × n_vars  │ ║
║ │ = 300 × 39,454                     │ = 26,230 × 351                      │ ║
║ │     obs: 'sex', 'age'              │     obs: 'orig.ident',              │ ║
║ │                                    │ 'nCount_RNA', 'nFeature_RNA',       │ ║
║ │                                    │ 'percent.mt', 'donor_id',           │ ║
║ │                                    │ 'pool_number',                      │ ║
║ │                                    │ 'predicted.celltype.l2',            │ ║
║ │                                    │ 'predicted.celltype.l2.score',      │ ║
║ │                                    │ 'age', 'organism_ontology_term_id', │ ║
║ │                                    │ 'tissue_ontology_term_id',          │ ║
║ │                                    │ 'assay_ontology_term_id',           │ ║
║ │                                    │ 'disease_ontology_term_id',         │ ║
║ │                                    │ 'cell_type_ontology_term_id',       │ ║
║ │                                    │ 'self_reported_ethnicity_ontology_… │ ║
║ │                                    │ 'development_stage_ontology_term_i… │ ║
║ │                                    │ 'sex_ontology_term_id',             │ ║
║ │                                    │ 'is_primary_data',                  │ ║
║ │                                    │ 'suspension_type', 'tissue_type',   │ ║
║ │                                    │ 'cell_type', 'assay', 'disease',    │ ║
║ │                                    │ 'organism', 'sex', 'tissue',        │ ║
║ │                                    │ 'self_reported_ethnicity',          │ ║
║ │                                    │ 'development_stage',                │ ║
║ │                                    │ 'observation_joinid'                │ ║
║ │     var: 'AF', 'AN', 'ER2',        │     var: 'vst.mean',                │ ║
║ │ 'IMPUTED', 'maf', 'NS', 'R2',      │ 'vst.variance',                     │ ║
║ │ 'TYPED', 'TYPED_ONLY', 'id',       │ 'vst.variance.expected',            │ ║
║ │ 'id_mask', 'length', 'pos',        │ 'vst.variance.standardized',        │ ║
║ │ 'quality', 'a0', 'a1', 'chrom',    │ 'vst.variable',                     │ ║
║ │ 'pos_hg19', 'id_hg19'              │ 'feature_is_filtered',              │ ║
║ │                                    │ 'feature_name',                     │ ║
║ │                                    │ 'feature_reference',                │ ║
║ │                                    │ 'feature_biotype',                  │ ║
║ │                                    │ 'feature_length', 'feature_type',   │ ║
║ │                                    │ 'start', 'end', 'chrom', 'strand'   │ ║
║ │     uns: 'dummy_onek1k',           │     uns:                            │ ║
║ │ 'kinship',                         │ 'cell_type_ontology_term_id_colors… │ ║
║ │ 'variant_annotation_vep'           │ 'citation', 'default_embedding',    │ ║
║ │                                    │ 'schema_reference',                 │ ║
║ │                                    │ 'schema_version', 'title', 'log1p'  │ ║
║ │     obsm: 'gPCs', 'ePCs', 'PB_CD4  │     obsm: 'X_azimuth_spca',         │ ║
║ │ Naive'                             │ 'X_azimuth_umap', 'X_harmony',      │ ║
║ │                                    │ 'X_pca', 'X_umap'                   │ ║
║ │     varm: 'variant_annotation'     │     varm: 'PCs'                     │ ║
║ └────────────────────────────────────┴─────────────────────────────────────┘ ║
╚═ group_id='donor_id' ════════════════════════════════════════════════════════╝

### Custom MAF Weights

We add custom MAF weights commonly used in burden tests, such as `Beta(MAF, 1, 25)`. These weights prioritize rarer variants in the analysis. TSS distance weight as used in the SAIGE-QTL paper are added manually for each gene

In [22]:
hd.G.varm["variant_annotation"]["maf_beta"] = beta_weighting(hd.G.var["maf"])

### Run burden tests using each annotation individually for weighting

Next, we specify the covariate matrix `F`, which includes sex, age, genetic PCs, and expression PCs. This controls for potential confounders in the association analysis.
We then iterate over all genes on chromosome 22, define a cis-window of ±500kb, and run a burden test using each of the annotation-based weighting schemes.

In [23]:
# This specifies covariates/fixed effects
F = np.concatenate(
    [
        np.ones((hd.shape[0], 1)),
        hd.G.obs[["sex"]].values - 1,
        hd.G.obs[["age"]].values,
        hd.G.obsm["gPCs"].values,
        hd.G.obsm["ePCs"],
    ],
    axis=1,
).astype(np.float64)
F[:, 2:] = column_normalize(F[:, 2:])

In [24]:
results = []
if isinstance(hd.G.X, da.Array | ad._core.views.DaskArrayView):
    if hd.G.is_view:
        hd._G = hd._G.copy()
    hd.G.X = hd.G.X.compute()

if do_debug:
    warnings.filterwarnings("ignore", category=RuntimeWarning)

for gene, row in tqdm(hd.C.var.iterrows(), total=hd.shape[3]):
    Y = gaussianize(hd.G.obsm[pb_gex_key][[gene]].values.astype(np.float64) + 1e-5 * np.random.randn(hd.shape[0], 1))

    start = max(0, row.start - cis_window)
    end = row.end + cis_window
    _G = hd.G[:, (hd.G.var.pos < end)]
    _G = _G[:, (_G.var.pos > start)]
    _G = _G[:, (_G.X.std(0) != 0)]
    _G = _G.copy()

    # TODO make strand aware
    _G.varm["variant_annotation"]["tss_distance"] = np.abs(row.start - _G.var["pos"])
    _G.varm["variant_annotation"]["tss_distance_exp"] = np.exp(-1e-5 * _G.varm["variant_annotation"]["tss_distance"])

    rdf = run_burden_test(
        _G, Y, F, gene, annotation_cols=annotation_cols, burden_agg_fct=burden_agg_fct, run_lrt=run_lrt
    )
    results.append(rdf)

rdf = pd.concat(results)
rdf

  0%|          | 0/351 [00:00<?, ?it/s]

site-packages/cellink/at/gwas.py:141: RuntimeWarning: divide by zero encountered in divide
  n = 1.0 / (GG - np.einsum("ij,ij->j", FG, A0iFG))
site-packages/cellink/at/gwas.py:142: RuntimeWarning: invalid value encountered in multiply
  M = -n * A0iFG
site-packages/cellink/at/gwas.py:146: RuntimeWarning: invalid value encountered in multiply
  self.beta_g += n[:, None] * GY


,burden_gene,egene,weight_col,burden_agg_fct,pv,beta,betaste,lrt
0,ENSG00000177663,ENSG00000177663,CADD_RAW,sum,0.271773,-1.720156e-02,1.565217e-02,1.207777
1,ENSG00000177663,ENSG00000177663,maf_beta,sum,0.793630,-8.648211e-05,3.305899e-04,0.068434
2,ENSG00000177663,ENSG00000177663,tss_distance,sum,0.516764,-3.056441e-08,4.714239e-08,0.420348
3,ENSG00000177663,ENSG00000177663,tss_distance_exp,sum,0.761835,-2.191842e-03,7.232076e-03,0.091853
0,ENSG00000069998,ENSG00000069998,CADD_RAW,sum,0.661393,-1.378342e-02,3.146971e-02,0.191835
...,...,...,...,...,...,...,...,...
3,ENSG00000100299,ENSG00000100299,tss_distance_exp,sum,0.380869,-2.700729e-03,3.081980e-03,0.767896
0,ENSG00000079974,ENSG00000079974,CADD_RAW,sum,NaN,NaN,NaN,NaN
1,ENSG00000079974,ENSG00000079974,maf_beta,sum,0.394131,-1.177792e-03,1.382141e-03,0.726160
2,ENSG00000079974,ENSG00000079974,tss_distance,sum,0.195245,-3.955153e-07,3.053661e-07,1.677586


## Step 3: Combine Multiple Tests with ACAT

To summarize evidence from multiple annotations, we use the **ACAT (Aggregated Cauchy Association Test)**. This meta-analysis method combines p-values from the burden tests per gene into a single statistic, improving power and interpretability.

In [25]:
combined = rdf.dropna(subset=["pv"]).groupby("egene")["pv"].agg(lambda x: acat_test(x.values)).reset_index()
combined.sort_values("pv")

,egene,pv
72,ENSG00000100154,0.000624
152,ENSG00000128191,0.001715
312,ENSG00000261188,0.002435
289,ENSG00000239713,0.002865
80,ENSG00000100219,0.003201
...,...,...
292,ENSG00000241360,0.992065
100,ENSG00000100294,0.994148
139,ENSG00000100410,0.997425
203,ENSG00000183741,0.997975


## Step 4: SKAT

As an alternative to burden testing, we apply the Sequence Kernel Association Test (SKAT), which models the variance component of aggregated rare variants rather than assuming a unidirectional effect. SKAT is more robust when variant effects differ in direction or magnitude.

Currently, `cellink` supports SKAT with the standard Beta(MAF, 1, 25) weighting scheme.

In [26]:
import logging

logger = logging.getLogger()
logger.setLevel(logging.WARNING)  # Suppress INFO and DEBUG esp. from SKAT Test

In [27]:
results = []

for gene, row in tqdm(hd.C.var.iterrows(), total=hd.shape[3]):
    Y = gaussianize(hd.G.obsm[pb_gex_key][[gene]].values.astype(np.float64) + 1e-5 * np.random.randn(hd.shape[0], 1))

    start = max(0, row.start - cis_window)
    end = row.end + cis_window
    _G = hd.G[:, (hd.G.var.pos < end)]
    _G = _G[:, (_G.var.pos > start)]
    _G = _G[:, (_G.X.std(0) != 0)]

    rdict = run_skat_test(_G, Y, F, gene)
    results.append(rdict)

rdf = pd.DataFrame(results)
rdf

  0%|          | 0/351 [00:00<?, ?it/s]

site-packages/limix_core/__init__.py:12: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound as _DistributionNotFound


site-packages/chiscore/_davies.py:133: RuntimeWarning: invalid value encountered in scalar divide
  s1 = c1[2] / c1[1] ** (3 / 2)
site-packages/chiscore/_davies.py:134: RuntimeWarning: invalid value encountered in scalar divide
  s2 = c1[3] / c1[1] ** 2
site-packages/chiscore/_davies.py:112: RuntimeWarning: invalid value encountered in divide
  Q_Norm = (Q_all - param["muQ"]) / param["sigmaQ"]


,burden_gene,egene,weight_col,pv
0,ENSG00000177663,ENSG00000177663,maf_beta,[[0.3131976285838407]]
1,ENSG00000069998,ENSG00000069998,maf_beta,[[0.5160870473428338]]
2,ENSG00000185837,ENSG00000185837,maf_beta,[[0.8831923053120991]]
3,ENSG00000093072,ENSG00000093072,maf_beta,[[0.9922485432501504]]
4,ENSG00000131100,ENSG00000131100,maf_beta,[[0.9212466195762082]]
...,...,...,...,...
346,ENSG00000205560,ENSG00000205560,maf_beta,[[0.39531531950768695]]
347,ENSG00000100288,ENSG00000100288,maf_beta,[[0.6082269209910665]]
348,ENSG00000205559,ENSG00000205559,maf_beta,[[0.09628933764662606]]
349,ENSG00000100299,ENSG00000100299,maf_beta,[[0.20923518926264417]]


In [28]:
rdf

,burden_gene,egene,weight_col,pv
0,ENSG00000177663,ENSG00000177663,maf_beta,[[0.3131976285838407]]
1,ENSG00000069998,ENSG00000069998,maf_beta,[[0.5160870473428338]]
2,ENSG00000185837,ENSG00000185837,maf_beta,[[0.8831923053120991]]
3,ENSG00000093072,ENSG00000093072,maf_beta,[[0.9922485432501504]]
4,ENSG00000131100,ENSG00000131100,maf_beta,[[0.9212466195762082]]
...,...,...,...,...
346,ENSG00000205560,ENSG00000205560,maf_beta,[[0.39531531950768695]]
347,ENSG00000100288,ENSG00000100288,maf_beta,[[0.6082269209910665]]
348,ENSG00000205559,ENSG00000205559,maf_beta,[[0.09628933764662606]]
349,ENSG00000100299,ENSG00000100299,maf_beta,[[0.20923518926264417]]
